# NB11 v2 -- real vs. fake class presence_score, with segmentation image

Extends NB09's existing "Part D -- false-class robustness" test (already
built for this exact ask, comment credits "teammate DAN TUD"), which
mixes real + distractor (fake/absent) classes into one combined prompt
list and segments the tile. This notebook adds what Part D does not
produce: the literal `presence_score` scalar per class per crop (Part D
only saves the final argmax'd label map).

Tile: `dop20_32_476_5524_1_he`. Real classes (9, from Part D's own
config, exact same words/colors -- not retyped from memory): water,
building, road, pitch, court, trees, vehicles, grass, tracks.
Fake/distractor classes (9): Part D's proven 5 (aircraft, stadium,
shipping container, greenhouse, parking lot) plus 4 new ones (windmill,
elephant, ship, camel) -- none plausible on this water/sports/road/tree
tile.

`confidence_threshold=0.05`, `prob_thd=0.05`, `slide_crop=768`,
`slide_stride=576` (Part D's own tile-specific baseline, not NB11 v1's
1024/768, so results are directly comparable to Part D's existing
output).

Two passes: Pass 1 builds an actual segmented image (real+fake combined,
18-class legend, via NB09's own `render_result` -- same colors/layout as
Part D's own rendered output, not a reimplementation) -- the
segmentation output that v1 was missing. Pass 2 captures the literal
presence_score number per class per crop. 7 plots + a numeric summary
table/JSON, not just log lines.


## 1 — Environment setup

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /tmp/miniconda

os.environ.pop("PYTHONPATH", None)
os.environ["PATH"] = "/tmp/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/tmp/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 -q

In [ ]:
!conda run -n segearth pip install openmim -q
!conda run -n segearth mim install "mmcv==2.2.0" -q
!conda run -n segearth pip install "mmsegmentation==1.2.2" -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
f = pathlib.Path("/tmp/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py")
f.write_text(f.read_text().replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'"))
print("Patched MMCV_MAX \u2192 2.3.0")
EOF
pip install numpy==1.26.4 -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import mmcv; print("MMCV:", mmcv.__version__)
from mmseg.structures import SegDataSample; print("MMSEG OK")
import torch; print("CUDA:", torch.cuda.is_available())
EOF

## 2 — Clone our fork

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path("/tmp/SegEarth-OV-3")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    print(f"Updated \u2192 {REPO}")
else:
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/HarishDeepak/rg-segearth-ov3", str(REPO)],
        check=True)
    print(f"Cloned \u2192 {REPO}")

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

## 3 -- Inference: segmentation (real+fake combined) + presence_score capture

In [ ]:
%%bash
export MPLBACKEND=Agg
export PYTHONUNBUFFERED=1
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3

python - << 'PYEOF'
import sys, json, torch, torch.nn.functional as F
import numpy as np
from pathlib import Path
from PIL import Image

sys.stdout.reconfigure(line_buffering=True)

DEVICE   = "cuda"
OUT_DIR  = Path("/kaggle/working/output"); OUT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR = OUT_DIR / "preds"; PRED_DIR.mkdir(parents=True, exist_ok=True)
BG_IDX = 255

CONF_THD     = 0.05
PROB_THD     = 0.05
SLIDE_CROP   = 768
SLIDE_STRIDE = 576

STEM = "dop20_32_476_5524_1_he"

# Real classes: exact words/display/colors from NB09's Part D config for
# this tile (IMAGE_CONFIGS["dop20_32_476_5524_1_he"]), not retyped from
# memory -- copied verbatim so results are directly comparable.
REAL_WORDS = [
    "water body, river, lake",
    "building, rooftop, residential roof",
    "sunlit paved road with lane markings and curbs, street with passing cars, not shadow, not dark shaded area",
    "football pitch with painted field markings encircled by an oval running track, athletics track and field",
    "clay sports court, tennis court, dirt sports ground",
    "tree, wooded canopy",
    "car, vehicle",
    "grass, lawn, low vegetation",
    "railway",
]
REAL_DISPLAY = ["water", "building", "road", "pitch", "court", "trees", "vehicles", "grass", "tracks"]
REAL_COLORS = [
    [0, 100, 255],
    [0, 0, 150],
    [80, 80, 80],
    [0, 200, 0],
    [180, 100, 40],
    [34, 139, 34],
    [255, 255, 0],
    [0, 255, 255],
    [128, 0, 128],
]

# Fake/distractor classes: Part D's own proven 5 (aircraft = confirmed
# clean negative control) plus 4 new ones per follow-up request.
FAKE_WORDS = ["aircraft", "stadium", "shipping container", "greenhouse", "parking lot",
              "windmill", "elephant", "ship", "camel"]
FAKE_DISPLAY = ["aircraft", "stadium", "container", "greenhouse", "parking lot",
                "windmill", "elephant", "ship", "camel"]
FAKE_COLORS = [
    [255, 20, 147],
    [139, 0, 0],
    [255, 215, 0],
    [0, 128, 128],
    [192, 192, 192],
    [255, 105, 180],
    [128, 64, 0],
    [30, 144, 255],
    [210, 180, 140],
]

ALL_WORDS = REAL_WORDS + FAKE_WORDS
ALL_DISPLAY = REAL_DISPLAY + FAKE_DISPLAY
ALL_COLORS = REAL_COLORS + FAKE_COLORS
N_REAL = len(REAL_WORDS)

def find_tile(stem):
    hits = sorted(Path("/kaggle/input").rglob(f"{stem}.jpg"))
    return hits[0] if hits else None

img_path = find_tile(STEM)
print(f"Resolved {STEM} -> {img_path}", flush=True)
if img_path is None:
    print("ERROR: tile not found under /kaggle/input.", flush=True)
    raise SystemExit(1)

from config_local import SAM3_CHECKPOINT
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

print("Loading SAM3...", flush=True)
model = build_sam3_image_model(
    bpe_path="./sam3/assets/bpe_simple_vocab_16e6.txt.gz",
    checkpoint_path=SAM3_CHECKPOINT, device=DEVICE)
model.eval()
for p in model.parameters(): p.requires_grad = False
print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)

def make_processor(conf_thd):
    return Sam3Processor(model, confidence_threshold=conf_thd, device=DEVICE)

def cache_text(processor, words):
    cache = []
    with torch.no_grad():
        for word in words:
            te = model.backbone.forward_text([word], device=DEVICE)
            cache.append({k: v.cpu() for k, v in te.items()})
    return cache

def collect_class_scores(processor, state, h, w, te_cache, n_classes, device):
    logits = torch.zeros((n_classes, h, w), device=device)
    for cls_idx, te_cpu in enumerate(te_cache):
        processor.reset_all_prompts(state)
        for k, v in te_cpu.items(): state["backbone_out"][k] = v.to(device)
        state["geometric_prompt"] = model._get_dummy_prompt()
        processor._forward_grounding(state)
        scores = torch.zeros((h, w), device=device)
        if state.get("masks_logits") is not None and state["masks_logits"].shape[0] > 0:
            for i in range(state["masks_logits"].shape[0]):
                il = state["masks_logits"][i].squeeze()
                if il.shape != (h, w):
                    il = F.interpolate(il.view(1,1,*il.shape), size=(h,w),
                                       mode="bilinear", align_corners=False).squeeze()
                scores = torch.max(scores, il * state["object_score"][i])
        sem = state["semantic_mask_logits"].squeeze()
        if sem.shape != (h, w):
            sem = F.interpolate(sem.view(1,1,*sem.shape), size=(h,w),
                                mode="bilinear", align_corners=False).squeeze()
        scores = torch.max(scores, sem) * state["presence_score"]
        logits[cls_idx] = torch.max(logits[cls_idx], scores)
    return logits

def make_gaussian_kernel(h, w, dev):
    sy, sx = h/4.0, w/4.0
    y = torch.arange(h, device=dev).float() - (h-1)/2.0
    x = torch.arange(w, device=dev).float() - (w-1)/2.0
    return torch.exp(-y[:,None]**2/(2*sy**2)) * torch.exp(-x[None,:]**2/(2*sx**2))

def run_sliding_window(img_arr, words, processor, crop_size, stride):
    te_cache = cache_text(processor, words)
    n_cls = len(words)
    H_full, W_full = img_arr.shape[:2]

    h_grids = max(H_full - crop_size + stride - 1, 0) // stride + 1
    w_grids = max(W_full - crop_size + stride - 1, 0) // stride + 1
    total = h_grids * w_grids

    gauss_k = make_gaussian_kernel(crop_size, crop_size, DEVICE)
    acc     = torch.zeros(n_cls, H_full, W_full, device=DEVICE)
    wt_mat  = torch.zeros(H_full, W_full, device=DEVICE)

    for hi in range(h_grids):
        for wi in range(w_grids):
            y1 = hi*stride;  x1 = wi*stride
            y2 = min(y1+crop_size, H_full);  x2 = min(x1+crop_size, W_full)
            y1 = max(y2-crop_size, 0);       x1 = max(x2-crop_size, 0)

            crop_pil = Image.fromarray(img_arr[y1:y2, x1:x2])
            h_c, w_c = y2-y1, x2-x1

            with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
                state = processor.set_image(crop_pil)
                l = collect_class_scores(processor, state, h_c, w_c, te_cache, n_cls, DEVICE).float()

            g = gauss_k[:h_c, :w_c]
            acc[:, y1:y2, x1:x2] += l * g.unsqueeze(0)
            wt_mat[y1:y2, x1:x2] += g

            done = hi*w_grids + wi + 1
            print(f"    crop {done}/{total}", flush=True)

    return acc / wt_mat.unsqueeze(0)

def finalize(prob_map, prob_thd, bg_idx=BG_IDX):
    seg = prob_map.argmax(0)
    seg[prob_map.max(0)[0] < prob_thd] = bg_idx
    return seg.cpu().numpy()

def collect_presence_scores(processor, state, te_cache):
    """Like collect_class_scores, but returns presence_score per class
    instead of a per-pixel logit map -- captures the scalar gate value
    before it gets multiplied into per-pixel scores and discarded."""
    scores = []
    for te_cpu in te_cache:
        processor.reset_all_prompts(state)
        for k, v in te_cpu.items(): state["backbone_out"][k] = v.to(DEVICE)
        state["geometric_prompt"] = model._get_dummy_prompt()
        processor._forward_grounding(state)
        ps = state["presence_score"]
        scores.append(float(ps.item()) if hasattr(ps, "item") else float(ps))
    return scores

img_arr = np.array(Image.open(img_path).convert("RGB"))
img_size = (img_arr.shape[1], img_arr.shape[0])

# ── Pass 1: segmentation (Part-D-style) -- real+fake combined, one argmax'd map ──
print("\n=== Pass 1: segmentation (real+fake combined) ===", flush=True)
proc = make_processor(CONF_THD)
logits = run_sliding_window(img_arr, ALL_WORDS, proc, SLIDE_CROP, SLIDE_STRIDE)
seg = finalize(logits, PROB_THD)
np.save(str(PRED_DIR / f"{STEM}_seg.npy"), seg.astype(np.uint8))
print("  Saved segmentation.", flush=True)

# ── Pass 2: presence_score capture -- same 18 classes, per crop ──
print("\n=== Pass 2: presence_score capture ===", flush=True)
H_full, W_full = img_arr.shape[:2]
h_grids = max(H_full - SLIDE_CROP + SLIDE_STRIDE - 1, 0) // SLIDE_STRIDE + 1
w_grids = max(W_full - SLIDE_CROP + SLIDE_STRIDE - 1, 0) // SLIDE_STRIDE + 1
total = h_grids * w_grids
print(f"Grid: {h_grids}x{w_grids} = {total} crops", flush=True)

te_cache = cache_text(proc, ALL_WORDS)
per_crop_scores = {word: [] for word in ALL_DISPLAY}

crop_i = 0
for hi in range(h_grids):
    for wi in range(w_grids):
        y1 = hi*SLIDE_STRIDE;  x1 = wi*SLIDE_STRIDE
        y2 = min(y1+SLIDE_CROP, H_full);  x2 = min(x1+SLIDE_CROP, W_full)
        y1 = max(y2-SLIDE_CROP, 0);       x1 = max(x2-SLIDE_CROP, 0)

        crop_pil = Image.fromarray(img_arr[y1:y2, x1:x2])

        with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
            state = proc.set_image(crop_pil)
            scores = collect_presence_scores(proc, state, te_cache)

        for name, s in zip(ALL_DISPLAY, scores):
            per_crop_scores[name].append(s)

        crop_i += 1
        print(f"    crop {crop_i}/{total}", flush=True)

(PRED_DIR / f"{STEM}_meta.json").write_text(json.dumps(dict(
    img_size=img_size, display=ALL_DISPLAY, colors=ALL_COLORS,
    n_real=N_REAL, real_display=REAL_DISPLAY, fake_display=FAKE_DISPLAY,
    conf_thd=CONF_THD, prob_thd=PROB_THD, slide_crop=SLIDE_CROP, slide_stride=SLIDE_STRIDE)))

(OUT_DIR / "presence_scores.json").write_text(json.dumps(dict(
    stem=STEM, conf_thd=CONF_THD, prob_thd=PROB_THD,
    slide_crop=SLIDE_CROP, slide_stride=SLIDE_STRIDE,
    real_display=REAL_DISPLAY, fake_display=FAKE_DISPLAY,
    per_crop_scores=per_crop_scores)))

print("\nPer-class presence_score summary:", flush=True)
for name in ALL_DISPLAY:
    vals = per_crop_scores[name]
    group = "real" if name in REAL_DISPLAY else "fake"
    print(f"  [{group}] {name}: min={min(vals):.4f} mean={sum(vals)/len(vals):.4f} max={max(vals):.4f}", flush=True)

print("\nWrote presence_scores.json and segmentation.", flush=True)
PYEOF


## 4 -- Rendering + plotting (GPU-free)

Renders the segmentation image (real+fake, 18-class legend, via NB09's
own render_result) plus 7 plots (2x bar+whiskers, faceted grouped bars,
single-axes color-coded bars, box plot, strip/jitter scatter, class x
crop heatmap) and a numeric summary table saved as JSON.


In [ ]:
import json
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

OUT_DIR  = Path("/kaggle/working/output")
PRED_DIR = OUT_DIR / "preds"
BG_IDX = 255

def find_tile(stem):
    hits = sorted(Path("/kaggle/input").rglob(f"{stem}.jpg"))
    return hits[0] if hits else None

# to_rgb / render_result copied verbatim from NB09's
# notebooks/push/nb09/NB09_zeroshot_sensitivity.ipynb cell 13 (re-synced
# from dummyirl/SegEarth-OV-3's segment.py) -- same segmentation image
# template/colors/layout as Part D's own output.

def to_rgb(seg, color_map, bg_idx=BG_IDX):
    out = np.zeros((*seg.shape, 3), dtype=np.uint8)
    safe = np.where(seg == bg_idx, 0, seg)
    out[:] = color_map[np.clip(safe, 0, len(color_map)-1)]
    out[seg == bg_idx] = [30, 30, 30]
    return out

def render_result(stem, img_arr, seg, color_map, display_labels, out_path,
                   img_size, prob_thd, conf_thd, slide_stride, slide_crop, alpha=0.5):
    fig, ax = plt.subplots(1, 2, figsize=(10, 7), dpi=300)
    fig.subplots_adjust(wspace=0)

    ax[0].imshow(img_arr)
    ax[0].axis('off')
    ax[0].set_title(f"{stem}.jpg", fontsize=10, fontweight='bold')

    ax[1].imshow(img_arr)
    ax[1].imshow(to_rgb(seg, color_map), alpha=alpha)
    ax[1].axis('off')
    ax[1].set_title(f'Segmentation Result (α={alpha})', fontsize=10, fontweight='bold')

    fig.tight_layout(rect=[0, 0.15, 1, 1])

    legend_elements = []
    for class_name, color in zip(display_labels, color_map):
        legend_elements.append(Patch(facecolor=color / 255.0, edgecolor='black', label=class_name))

    fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, 0.1),
               frameon=False, ncol=min(4, len(display_labels)), prop={'size': 9, 'weight': 'bold'})

    meta_text = (f"img_size = {img_size[0]}x{img_size[1]}    "
                 f"prob_thd = {prob_thd}    "
                 f"conf_thd = {conf_thd}\n"
                 f"slide_stride = {slide_stride}    "
                 f"slide_crop = {slide_crop}")
    fig.text(0.5, 0.025, meta_text, ha='center', va='bottom', fontsize=10, family='monospace')

    fig.savefig(str(out_path))
    plt.close(fig)
    print(f"  Rendered: {out_path.name}", flush=True)

meta = json.loads((PRED_DIR / "dop20_32_476_5524_1_he_meta.json").read_text())
STEM = "dop20_32_476_5524_1_he"
DISPLAY = meta["display"]
COLORS = np.array(meta["colors"], dtype=np.uint8)
REAL_DISPLAY = meta["real_display"]
FAKE_DISPLAY = meta["fake_display"]
img_size = tuple(meta["img_size"])

img_arr = np.array(Image.open(find_tile(STEM)).convert("RGB"))

# ── Segmentation image (Pass 1 output) ──
seg = np.load(str(PRED_DIR / f"{STEM}_seg.npy"))
render_result(STEM, img_arr, seg, COLORS, DISPLAY,
              OUT_DIR / f"{STEM}_real_fake_segmentation.png", img_size,
              meta["prob_thd"], meta["conf_thd"], meta["slide_stride"], meta["slide_crop"])

# ── Load presence_score data (Pass 2 output) ──
data = json.loads((OUT_DIR / "presence_scores.json").read_text())
PER_CROP = data["per_crop_scores"]

def stats(name):
    vals = np.array(PER_CROP[name])
    return vals.min(), vals.mean(), vals.max(), vals

real_stats = {n: stats(n) for n in REAL_DISPLAY}
fake_stats = {n: stats(n) for n in FAKE_DISPLAY}

# ── Numeric summary table (required output, not a plot byproduct) ──
summary = {"real": {}, "fake": {}}
print("\n=== presence_score numeric summary ===")
print(f"{'group':6} {'class':12} {'min':>8} {'mean':>8} {'max':>8}")
for group, group_stats in [("real", real_stats), ("fake", fake_stats)]:
    for name, (mn, mean, mx, _) in group_stats.items():
        summary[group][name] = dict(min=float(mn), mean=float(mean), max=float(mx))
        print(f"{group:6} {name:12} {mn:8.4f} {mean:8.4f} {mx:8.4f}")

(OUT_DIR / "presence_scores_summary.json").write_text(json.dumps(summary, indent=2))
print("\nWrote presence_scores_summary.json")

# ── Plot 1: bar chart w/ min-max whiskers, fake classes only ──
fig, ax = plt.subplots(figsize=(9, 6))
names = FAKE_DISPLAY
means = np.array([fake_stats[n][1] for n in names])
mins = np.array([fake_stats[n][0] for n in names])
maxs = np.array([fake_stats[n][2] for n in names])
x = np.arange(len(names))
ax.bar(x, means, yerr=[means - mins, maxs - means], capsize=6,
       color="tab:red", alpha=0.7, edgecolor="black")
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=10, rotation=20)
ax.set_ylabel("presence_score")
ax.set_ylim(0, max(0.05, maxs.max() * 1.3))
ax.set_title("Fake/absent classes: presence_score stays low\n"
             "(bar = mean, whiskers = observed min-max across crops)", fontsize=11)
fig.tight_layout()
fig.savefig(str(OUT_DIR / f"{STEM}_plot1_fake_bar_whiskers.png"), dpi=200)
plt.close(fig)
print("  Rendered: plot1_fake_bar_whiskers.png")

# ── Plot 2: bar chart w/ min-max whiskers, real classes only ──
fig, ax = plt.subplots(figsize=(9, 6))
names = REAL_DISPLAY
means = np.array([real_stats[n][1] for n in names])
mins = np.array([real_stats[n][0] for n in names])
maxs = np.array([real_stats[n][2] for n in names])
x = np.arange(len(names))
ax.bar(x, means, yerr=[means - mins, maxs - means], capsize=6,
       color="tab:green", alpha=0.7, edgecolor="black")
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=10, rotation=20)
ax.set_ylabel("presence_score")
ax.set_ylim(0, 1.0)
ax.set_title("Real/present classes: presence_score for what's\n"
             "actually in the scene (bar = mean, whiskers = observed min-max)", fontsize=11)
fig.tight_layout()
fig.savefig(str(OUT_DIR / f"{STEM}_plot2_real_bar_whiskers.png"), dpi=200)
plt.close(fig)
print("  Rendered: plot2_real_bar_whiskers.png")

# ── Plot 3: faceted grouped bar chart, real | fake side by side, shared y-axis ──
shared_max = max(
    max(real_stats[n][2] for n in REAL_DISPLAY),
    max(fake_stats[n][2] for n in FAKE_DISPLAY),
) * 1.2
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, names, group_stats, color, label in [
    (axes[0], REAL_DISPLAY, real_stats, "tab:green", "REAL classes"),
    (axes[1], FAKE_DISPLAY, fake_stats, "tab:red", "FAKE classes"),
]:
    means = np.array([group_stats[n][1] for n in names])
    mins = np.array([group_stats[n][0] for n in names])
    maxs = np.array([group_stats[n][2] for n in names])
    x = np.arange(len(names))
    ax.bar(x, means, yerr=[means - mins, maxs - means], capsize=5,
           color=color, alpha=0.7, edgecolor="black")
    ax.set_xticks(x); ax.set_xticklabels(names, fontsize=9, rotation=25)
    ax.set_title(label, fontsize=11)
    ax.set_ylim(0, shared_max)
axes[0].set_ylabel("presence_score")
fig.suptitle("Real vs. fake: presence_score comparison\n"
             "(real classes should score higher than fake ones)", fontsize=12)
fig.tight_layout()
fig.savefig(str(OUT_DIR / f"{STEM}_plot3_real_vs_fake_faceted.png"), dpi=200)
plt.close(fig)
print("  Rendered: plot3_real_vs_fake_faceted.png")

# ── Plot 4: single-axes bar chart, all 18 classes, color-coded by group ──
fig, ax = plt.subplots(figsize=(14, 6))
all_names = REAL_DISPLAY + FAKE_DISPLAY
all_stats = {**real_stats, **fake_stats}
means = np.array([all_stats[n][1] for n in all_names])
mins = np.array([all_stats[n][0] for n in all_names])
maxs = np.array([all_stats[n][2] for n in all_names])
colors = ["tab:green"] * len(REAL_DISPLAY) + ["tab:red"] * len(FAKE_DISPLAY)
x = np.arange(len(all_names))
ax.bar(x, means, yerr=[means - mins, maxs - means], capsize=4,
       color=colors, alpha=0.75, edgecolor="black")
ax.set_xticks(x); ax.set_xticklabels(all_names, fontsize=9, rotation=35, ha="right")
ax.set_ylabel("presence_score")
ax.legend(handles=[Patch(color="tab:green", label="real"), Patch(color="tab:red", label="fake")])
ax.set_title("Real vs. fake, all 18 classes on one chart\n"
             "(green = real/present, red = fake/absent)", fontsize=12)
fig.tight_layout()
fig.savefig(str(OUT_DIR / f"{STEM}_plot4_real_vs_fake_single_axes.png"), dpi=200)
plt.close(fig)
print("  Rendered: plot4_real_vs_fake_single_axes.png")

# ── Plot 5: box plot, real vs fake, one box per class ──
fig, ax = plt.subplots(figsize=(14, 6))
all_vals = [all_stats[n][3] for n in all_names]
bp = ax.boxplot(all_vals, tick_labels=all_names, patch_artist=True)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_xticklabels(all_names, fontsize=9, rotation=35, ha="right")
ax.set_ylabel("presence_score")
ax.legend(handles=[Patch(color="tab:green", label="real"), Patch(color="tab:red", label="fake")])
ax.set_title("presence_score distribution per class across all crops\n"
             "(box = IQR, orange line = median, dots = outlier crops)", fontsize=12)
fig.tight_layout()
fig.savefig(str(OUT_DIR / f"{STEM}_plot5_boxplot.png"), dpi=200)
plt.close(fig)
print("  Rendered: plot5_boxplot.png")

# ── Plot 6: strip/jitter scatter, one column per class ──
fig, ax = plt.subplots(figsize=(14, 6))
rng = np.random.default_rng(0)
for i, (name, color) in enumerate(zip(all_names, colors)):
    vals = all_stats[name][3]
    jitter = rng.uniform(-0.15, 0.15, size=len(vals))
    ax.scatter(np.full(len(vals), i) + jitter, vals, color=color, alpha=0.6, s=18, edgecolor="black", linewidth=0.3)
ax.set_xticks(range(len(all_names))); ax.set_xticklabels(all_names, fontsize=9, rotation=35, ha="right")
ax.set_ylabel("presence_score")
ax.legend(handles=[Patch(color="tab:green", label="real"), Patch(color="tab:red", label="fake")])
ax.set_title("presence_score per crop\n"
             "(each dot = one sliding-window crop, jittered horizontally to reduce overlap)", fontsize=12)
fig.tight_layout()
fig.savefig(str(OUT_DIR / f"{STEM}_plot6_strip_scatter.png"), dpi=200)
plt.close(fig)
print("  Rendered: plot6_strip_scatter.png")

# ── Plot 7: heatmap, class x crop grid ──
n_crops = len(all_stats[all_names[0]][3])
grid = np.array([all_stats[n][3] for n in all_names])  # (n_classes, n_crops)
fig, ax = plt.subplots(figsize=(max(10, n_crops * 0.3), 8))
im = ax.imshow(grid, aspect="auto", cmap="hot", vmin=0, vmax=max(0.1, grid.max()))
ax.set_yticks(range(len(all_names))); ax.set_yticklabels(all_names, fontsize=9)
ax.set_xlabel("crop index")
ax.axhline(len(REAL_DISPLAY) - 0.5, color="cyan", linewidth=2)
fig.colorbar(im, ax=ax, label="presence_score")
ax.set_title("presence_score heatmap -- real classes (top block) vs.\n"
             "fake classes (bottom block, below cyan line), all crops", fontsize=12)
fig.tight_layout()
fig.savefig(str(OUT_DIR / f"{STEM}_plot7_heatmap.png"), dpi=200)
plt.close(fig)
print("  Rendered: plot7_heatmap.png")

print("\nDone. Segmentation image + 7 plots + numeric summary produced.")
